# HW1 – 2025 – Simulation
## Domain: Video Games (PC & PlayStation)

**Author:** Georgios Kitsakis  
**Institution:** Athens University of Economics and Business (AUEB)

---

Synthetic dataset of video games rated by 5 user segments.
Ratings are binary (+1 like / -1 dislike), generated by rule-based criteria per segment.
LDA (tomotopy) with anchor words is used to recover the segments from the rating patterns.

The notebook runs in **two phases**:
- **Phase 1 — Simple criteria**: only the most obvious filter (platform / price / age rating)
- **Phase 2 — Full criteria**: adds Metacritic thresholds to make LIKE tokens more segment-specific

| # | Segment | Simple like condition | Full like condition |
|---|---|---|---|
| 1 | **PC Gamer** | platform in ('PC','BOTH') | + Metacritic ≥ 60 |
| 2 | **Console Gamer** | platform in ('PS','BOTH') | + Metacritic ≥ 55 |
| 3 | **Cross-Platform Gamer** | platform == 'BOTH' | + Metacritic ≥ 45 |
| 4 | **Budget Gamer** | price ≤ €25 | same (price is the key signal) |
| 5 | **Casual / Family Gamer** | PEGI '3'/'7' | same (age rating is already distinctive) |

In [ ]:
!pip install tomotopy -q

In [ ]:
import random
import csv
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Gamer',
    2: 'Console Gamer',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

---
## 1. `generate_entities()`

Each game has **10 distinct attributes**: title, platform, genre, price_eur, metacritic,
avg_playtime_h, is_multiplayer, is_exclusive, age_rating, release_year.

In [ ]:
@dataclass
class VideoGame:
    title: str
    platform: str          # 'PC', 'PS', or 'BOTH'
    genre: str
    price_eur: float
    metacritic: int        # 0-100
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool     # True when platform != 'BOTH'
    age_rating: str        # PEGI: '3','7','12','16','18'
    release_year: int

In [ ]:
def generate_entities(
    game_num: int = 200,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.35, 0.30, 0.35],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (2010, 2024)
) -> List[VideoGame]:
    """
    Generates a list of synthetic video game entities.

    Each game has 10 distinct attributes:
        title           - unique identifier (e.g. 'Game_0001')
        platform        - 'PC', 'PS', or 'BOTH' (sampled from platform_distro)
        genre           - one of genre_options
        price_eur       - Gaussian(35, 18), clipped to [5, 80]
        metacritic      - Gaussian(68, 15), clipped to [0, 100]
        avg_playtime_h  - Gaussian(25, 20), clipped to [1, 200]
        is_multiplayer  - True with probability multiplayer_prob
        is_exclusive    - True when platform != 'BOTH'
        age_rating      - PEGI rating: '3', '7', '12', '16', or '18'
        release_year    - uniform in year_range

    Args:
        game_num (int):                    Number of games. Default: 200.
        genre_options (List[str]):         Available genres.
        platform_options (List[str]):      Platform choices.
        platform_distro (List[float]):     Probability distribution over platforms.
        price_gaussian_params (Tuple):     (mean, std) for price in EUR.
        metacritic_gaussian_params (Tuple):(mean, std) for Metacritic score.
        playtime_gaussian_params (Tuple):  (mean, std) for avg playtime in hours.
        multiplayer_prob (float):          Probability of multiplayer support.
        age_ratings (List[str]):           PEGI rating options.
        year_range (Tuple[int, int]):      (min_year, max_year) for release year.

    Returns:
        List[VideoGame]: Generated game objects.
    """
    games = []
    for i in range(game_num):
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=f'Game_{i+1:04d}', platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [ ]:
games = generate_entities(game_num=200)
print(f'Generated {len(games)} games.')
print(vars(games[0]))
print('Platform distribution:', pd.Series([g.platform for g in games]).value_counts().to_dict())

---
## 2. `generate_users()`

One helper per segment, all called from `generate_users()`.

In [ ]:
@dataclass
class User:
    segment: int
    age: int
    gender: str

In [ ]:
def generate_users_segment1(user_num: int = 200) -> List[User]:
    """
    Segment 1 - PC Gamer.
    Dedicated PC-first players who prefer high-quality games.
    Age: Gaussian(30, 6).
    """
    return [User(segment=1, age=max(10, int(random.gauss(30, 6))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 200) -> List[User]:
    """
    Segment 2 - Console Gamer.
    PlayStation loyalists; buy any well-reviewed PS or cross-platform title.
    Age: Gaussian(25, 7).
    """
    return [User(segment=2, age=max(10, int(random.gauss(25, 7))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 200) -> List[User]:
    """
    Segment 3 - Cross-Platform Gamer.
    Own both PC and PlayStation; only buy games available on BOTH.
    Age: Gaussian(22, 5).
    """
    return [User(segment=3, age=max(10, int(random.gauss(22, 5))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 200) -> List[User]:
    """
    Segment 4 - Budget Gamer.
    Price-first players; any platform, any genre, as long as it is cheap.
    Age: Gaussian(20, 8).
    """
    return [User(segment=4, age=max(10, int(random.gauss(20, 8))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 200) -> List[User]:
    """
    Segment 5 - Casual / Family Gamer.
    Occasional players who want short, family-friendly games (PEGI 3/7).
    Age: Gaussian(38, 10).
    """
    return [User(segment=5, age=max(10, int(random.gauss(38, 10))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]

In [ ]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a shuffled population of users split equally across 5 segments.

    Calls:
        generate_users_segment1()  PC Gamer
        generate_users_segment2()  Console Gamer
        generate_users_segment3()  Cross-Platform Gamer
        generate_users_segment4()  Budget Gamer
        generate_users_segment5()  Casual / Family Gamer

    Args:
        user_num (int): Total users (user_num // 5 per segment). Default: 1000.

    Returns:
        List[User]: Shuffled User objects tagged with segment id (1-5).
    """
    n = user_num // 5
    users = (
        generate_users_segment1(n) +
        generate_users_segment2(n) +
        generate_users_segment3(n) +
        generate_users_segment4(n) +
        generate_users_segment5(n)
    )
    random.shuffle(users)
    return users

In [ ]:
users = generate_users(user_num=1000)
counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {cnt} users')

---
## 3. `generate_ratings()`

Each segment has its own helper. A `mode` parameter switches between **simple** and **full** criteria.
After the ground-truth rating is decided, it is flipped with probability `noise`.

In [ ]:
def _make_row(user, game, rating, reason):
    """Helper: flattens a user-game pair into a CSV row dict."""
    return {'segment': user.segment, 'age': user.age, 'gender': user.gender,
            'game': game.title, 'platform': game.platform, 'genre': game.genre,
            'price_eur': game.price_eur, 'metacritic': game.metacritic,
            'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
            'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
            'release_year': game.release_year, 'rating': rating, 'reason': reason}

In [ ]:
def generate_ratings_segment1(users, games, pairs, noise, mode):
    """
    Segment 1 - PC Gamer.

    Simple : Like if platform in ('PC', 'BOTH').
    Full   : Like if platform in ('PC', 'BOTH') AND Metacritic >= 60.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PC', 'BOTH'):
            rating, reason = -1, 'Not on PC'
        elif mode == 'simple':
            rating, reason = 1, 'PC platform'
        elif game.metacritic >= 60:
            rating, reason = 1, 'PC/BOTH + good Metacritic'
        else:
            rating, reason = -1, 'PC but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment2(users, games, pairs, noise, mode):
    """
    Segment 2 - Console Gamer.

    Simple : Like if platform in ('PS', 'BOTH').
    Full   : Like if platform in ('PS', 'BOTH') AND Metacritic >= 55.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PS', 'BOTH'):
            rating, reason = -1, 'Not on PlayStation'
        elif mode == 'simple':
            rating, reason = 1, 'PS platform'
        elif game.metacritic >= 55:
            rating, reason = 1, 'PS/BOTH + acceptable Metacritic'
        else:
            rating, reason = -1, 'PS platform but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment3(users, games, pairs, noise, mode):
    """
    Segment 3 - Cross-Platform Gamer.

    Simple : Like if platform == 'BOTH'.
    Full   : Like if platform == 'BOTH' AND Metacritic >= 45.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform != 'BOTH':
            rating, reason = -1, 'Not on both platforms'
        elif mode == 'simple':
            rating, reason = 1, 'BOTH platform'
        elif game.metacritic >= 45:
            rating, reason = 1, 'BOTH + acceptable Metacritic'
        else:
            rating, reason = -1, 'BOTH but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment4(users, games, pairs, noise, mode):
    """
    Segment 4 - Budget Gamer.

    Simple & Full: Like if price_eur <= 25 (price is the dominant signal).
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.price_eur <= 25:
            rating, reason = 1, 'Cheap enough'
        else:
            rating, reason = -1, 'Too expensive'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment5(users, games, pairs, noise, mode):
    """
    Segment 5 - Casual / Family Gamer.

    Simple & Full: Like if age_rating in ('3', '7').
    Age rating is already a strong segment-specific signal.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.age_rating in ('3', '7'):
            rating, reason = 1, 'Family age rating'
        else:
            rating, reason = -1, 'Age rating too high'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.05,
    output_file: str = 'ratings.csv',
    mode: str = 'full'
) -> None:
    """
    Generates binary like (+1) / dislike (-1) ratings and writes them to CSV.

    Like condition per segment (mode='full'):
        Segment 1 - PC Gamer           : PC/BOTH + Metacritic >= 60
        Segment 2 - Console Gamer      : PS/BOTH + Metacritic >= 55
        Segment 3 - Cross-Platform     : BOTH + Metacritic >= 45
        Segment 4 - Budget Gamer       : price <= 25 EUR
        Segment 5 - Casual/Family      : PEGI '3' or '7'

    With mode='simple' only the platform/price/age filter is applied (no Metacritic threshold).
    Ratings are flipped with probability `noise`.

    Calls generate_ratings_segment1() through generate_ratings_segment5().

    Args:
        users (List[User]):       User objects.
        games (List[VideoGame]):  VideoGame objects.
        n_ratings (int):          Number of (user, game) pairs. Default: 10000.
        noise (float):            Rating flip probability. Default: 0.05.
        output_file (str):        CSV output path. Default: 'ratings.csv'.
        mode (str):               'simple' or 'full'. Default: 'full'.

    Returns:
        None - writes CSV to output_file.
    """
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {
        1: generate_ratings_segment1,
        2: generate_ratings_segment2,
        3: generate_ratings_segment3,
        4: generate_ratings_segment4,
        5: generate_ratings_segment5,
    }

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise, mode))
    random.shuffle(all_rows)

    fieldnames = ['segment', 'age', 'gender', 'game', 'platform', 'genre', 'price_eur',
                  'metacritic', 'avg_playtime_h', 'is_multiplayer', 'is_exclusive',
                  'age_rating', 'release_year', 'rating', 'reason']
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Mode              : {mode}')
    print(f'Total ratings     : {total:,}')
    print(f'Positive (like)   : {positive:,}  ({positive/total:.1%})')
    print(f'Negative (dislike): {total-positive:,}  ({(total-positive)/total:.1%})')
    print(f'Noise level       : {noise:.0%}')
    print(f'Saved to          : {output_file}')

---
## 4. `learn_segments()` - LDA with Anchor Words

Each user becomes a **document**. Each rated game becomes a **token**: `Game_0042_PC_Strategy_LIKE`

**Anchor words** nudge each LDA topic toward the expected segment:
- Topic 0: `_PC_` tokens → PC Gamer
- Topic 1: `_PS_` tokens → Console Gamer
- Topic 2: `_BOTH_` tokens → Cross-Platform Gamer
- Topic 3: cheap-game LIKE tokens → Budget Gamer
- Topic 4: family-rated LIKE tokens → Casual / Family Gamer

In [ ]:
def learn_segments(
    ratings_file: str = 'ratings.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> None:
    """
    Discovers customer segments from rating patterns using LDA (tomotopy).

    Method:
        1. Each user -> one document; each rated game -> token
           '<title>_<platform>_<genre>_LIKE/DISLIKE'.
        2. LDA trained with k topics and anchor words per topic:
           Topic 0: _PC_ tokens       -> PC Gamer
           Topic 1: _PS_ tokens       -> Console Gamer
           Topic 2: _BOTH_ tokens     -> Cross-Platform Gamer
           Topic 3: cheap LIKE tokens -> Budget Gamer
           Topic 4: family LIKE tokens-> Casual / Family Gamer
        3. Top-N words per topic and cross-tab (true segment vs LDA topic) are printed.

    Args:
        ratings_file (str):      Path to the CSV from generate_ratings().
        games (List[VideoGame]): Game objects for anchor word lookup.
        k (int):                 Number of LDA topics. Default: 5.
        n_iter (int):            Training iterations. Default: 500.
        top_n_words (int):       Top words to show per topic. Default: 10.

    Returns:
        None - prints topic summaries and cross-tab.
    """
    df = pd.read_csv(ratings_file)
    game_lookup = {g.title: g for g in games} if games else {}

    # Build one document per user
    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [
            f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
            for _, r in grp.iterrows()
        ]
        if tokens:
            user_docs[uid] = tokens

    print(f'Documents (unique users) : {len(user_docs)}')
    print(f'Avg tokens per document  : {np.mean([len(v) for v in user_docs.values()]):.1f}')

    # Build anchor word lists (up to 20 per topic)
    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            parts = t.split('_')
            g_title = parts[0] + '_' + parts[1]
            if g_title not in game_lookup:
                continue
            g = game_lookup[g_title]
            if g.price_eur <= 25 and t.endswith('LIKE'):
                anchor_budget.append(t)
            if g.age_rating in ('3', '7') and t.endswith('LIKE'):
                anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'\nAnchor sizes: PC={len(anchor_pc)}, PS={len(anchor_ps)}, '
          f'BOTH={len(anchor_both)}, Budget={len(anchor_budget)}, Casual={len(anchor_casual)}')

    # Train LDA
    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)

    print('\nTraining LDA...')
    for i in range(0, n_iter, 50):
        lda.train(50)
        print(f'  Iteration {i+50:4d}  |  log-likelihood: {lda.ll_per_word:.4f}')

    # Print topic summaries
    print('\n' + '='*65)
    print('  LDA DISCOVERED TOPICS')
    print('='*65)
    for tid in range(lda.k):
        top_words  = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c   = sum(1 for w in top_words if '_PC_' in w)
        ps_c   = sum(1 for w in top_words if '_PS_' in w)
        bot_c  = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dis_c  = sum(1 for w in top_words if w.endswith('_DISLIKE'))
        dominant = max({'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c},
                       key=lambda x: {'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c}[x])
        genres    = [w.split('_')[3] for w in top_words if len(w.split('_')) >= 5]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'\n--- Topic {tid} ---')
        print(f'  Dominant platform : {dominant}  (PC={pc_c}, PS={ps_c}, BOTH={bot_c})')
        print(f'  Top genre         : {top_genre}')
        print(f'  Sentiment         : {like_c} LIKE vs {dis_c} DISLIKE')
        print(f'  Top tokens:')
        for w in top_words:
            print(f'    {w}')

    # Cross-tab: true segment vs most-probable LDA topic
    print('\n' + '='*65)
    print('  True Segment vs Most-Probable LDA Topic')
    print('='*65)
    uid_list = list(user_docs.keys())
    rows = [
        {'true_segment': int(uid_list[i].split('_')[0]),
         'lda_topic': int(np.argmax(doc.get_topic_dist()))}
        for i, doc in enumerate(lda.docs)
    ]
    ct = pd.crosstab(
        pd.DataFrame(rows)['true_segment'],
        pd.DataFrame(rows)['lda_topic'],
        rownames=['True Segment'], colnames=['LDA Topic']
    )
    print(ct)
    print('\nDone.')

---
---
# Phase 1 - Simple Criteria

Only the most obvious filter per segment:
- PC Gamer → any PC/BOTH game
- Console Gamer → any PS/BOTH game
- Cross-Platform → any BOTH game
- Budget → any game priced ≤ €25
- Casual → any game with PEGI 3/7

**Expectation:** Platform segments (1, 2, 3) separate cleanly. Budget and Casual may blur
since cheap games and family-rated games exist across all platforms.

In [ ]:
print('=== Phase 1: SIMPLE criteria ===')
generate_ratings(users, games, n_ratings=10000, noise=0.05,
                 output_file='ratings_simple.csv', mode='simple')

df_simple = pd.read_csv('ratings_simple.csv')
print()
print('Like rate per segment (simple):')
for seg, grp in df_simple.groupby('segment'):
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

In [ ]:
print('=== Phase 1: LDA on SIMPLE ratings ===')
learn_segments(ratings_file='ratings_simple.csv', games=games, k=5, n_iter=500)

### Phase 1 - Observations

With simple criteria the platform split (PC / PS / BOTH) is immediately visible in the tokens.
LDA picks it up almost trivially for Segments 1, 2, 3.

Segments 4 (Budget) and 5 (Casual) are noisier because cheap games and family-rated games
are spread across all platforms — their documents share vocabulary with the platform segments.

**→ Phase 2 adds Metacritic thresholds to make every segment's LIKE tokens more segment-specific.**

---
---
# Phase 2 - Full (Harder) Criteria

Metacritic quality thresholds are added on top of the simple filters:
- PC Gamer → **+ Metacritic ≥ 60**
- Console Gamer → **+ Metacritic ≥ 55**
- Cross-Platform → **+ Metacritic ≥ 45**
- Budget → same (price is the dominant signal)
- Casual → same (age rating is already distinctive)

**Expectation:** Like rates settle around 30-50% per segment. The PC and Console LIKE
tokens are now filtered to higher-quality games, reducing overlap with the Cross-Platform
segment. The LDA cross-tab diagonal should strengthen compared to Phase 1.

In [ ]:
print('=== Phase 2: FULL criteria ===')
generate_ratings(users, games, n_ratings=10000, noise=0.05,
                 output_file='ratings_full.csv', mode='full')

df_full = pd.read_csv('ratings_full.csv')
print()
print('Like rate per segment (full):')
for seg, grp in df_full.groupby('segment'):
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

In [ ]:
print('=== Phase 2: LDA on FULL ratings ===')
learn_segments(ratings_file='ratings_full.csv', games=games, k=5, n_iter=500)

### Phase 2 - Observations

With full criteria:
- PC and Console Gamer LIKE tokens are now filtered to well-reviewed games, making
  their vocabulary more distinctive from the Cross-Platform segment.
- Budget and Casual segments retain their strong single-attribute signal (price / age rating).
- The LDA cross-tab diagonal is stronger than Phase 1.

**Key takeaway:** Even modest Metacritic thresholds reduce vocabulary overlap between
platform segments. Anchor words give LDA a head start by associating each topic
with the correct platform tokens from the start.

---
## Summary

| Step | Required function | Helpers called |
|---|---|---|
| 1 | `generate_entities(game_num)` | - |
| 2 | `generate_users(user_num)` | `generate_users_segment1/2/3/4/5()` |
| 3 | `generate_ratings(users, games, ...)` | `generate_ratings_segment1/2/3/4/5()` |
| 4 | `learn_segments(ratings_file, games, k)` | - |

| Phase | Mode | Key observation |
|---|---|---|
| 1 | `simple` | Platform segments (1-3) easily separated; Budget/Casual blur |
| 2 | `full` | Metacritic threshold reduces token overlap; stronger diagonal in cross-tab |